# 03 — Geospatial search

Upsert normalized listings into PostGIS, then radius-search with raw SQL.
Requires `docker compose up -d db` (and fresh DB if schema changed).

In [ ]:
from notification_rake.config import settings
from notification_rake.storage import search_within_radius
from notification_rake.workflow import run_pipeline

result = run_pipeline(notify=False)
print(
    f"Upserted {result.upserted} ({result.new} new) — "
    f"{result.in_radius} within radius"
)

search_within_radius(
    settings.database_url,
    lon=settings.default_lon,
    lat=settings.default_lat,
    radius_m=50_000,
)

In [ ]:
import pandas as pd

nearby = search_within_radius(
    settings.database_url,
    lon=settings.default_lon,
    lat=settings.default_lat,
    radius_m=50_000,
)
pd.DataFrame(nearby)

## Hasura + alerts

Track tables for GraphQL (`make run CMD=hasura_track`), then query `vehicle_listing` in the
[Hasura console](http://127.0.0.1:8080/console). Geo SQL function: `listings_within_radius`.

New listings trigger Gotify when `GOTIFY_TOKEN` is set — run `make run CMD=pipeline` with alerts on.